# AI-MLOps Stock Prediction — Exploration Notebook

This notebook demonstrates:
1. Data inspection and EDA
2. Feature distribution analysis
3. SHAP feature importance
4. Sentiment analysis results
5. Chronos-2 zero-shot forecasting demo
6. Model comparison visualisation

> Run `python main.py --mode full-pipeline --trainer manual` before opening this notebook.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

SYMBOLS = cfg['stocks']['symbols']
SIGNAL_NAMES = ['Strong Sell', 'Sell', 'Hold', 'Buy', 'Strong Buy']
print('Config loaded. Symbols:', SYMBOLS)

## 1. Load Processed Features

In [ ]:
from src.data.features import load_processed

dfs = {}
for sym in SYMBOLS:
    try:
        dfs[sym] = load_processed(sym)
        print(f'{sym}: {len(dfs[sym])} rows × {len(dfs[sym].columns)} features')
    except FileNotFoundError:
        print(f'{sym}: not found — run pipeline first')

## 2. Label Distribution

In [ ]:
fig, axes = plt.subplots(1, len(dfs), figsize=(4 * len(dfs), 4))
if len(dfs) == 1:
    axes = [axes]

for ax, (sym, df) in zip(axes, dfs.items()):
    counts = df['label'].value_counts().sort_index()
    ax.bar([SIGNAL_NAMES[i] for i in counts.index], counts.values,
           color=['#d62728','#ff7f0e','#1f77b4','#2ca02c','#17becf'])
    ax.set_title(sym)
    ax.set_xlabel('Signal')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Label Distribution by Stock', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Feature Correlation (first symbol)

In [ ]:
if dfs:
    sym = list(dfs.keys())[0]
    df = dfs[sym]
    # Select key features
    key_feats = ['log_return_1d','rsi_14','macd','bb_width','atr_14',
                 'obv','sentiment_score','volume_sma_ratio','label']
    key_feats = [c for c in key_feats if c in df.columns]
    corr = df[key_feats].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
                square=True, ax=ax)
    ax.set_title(f'{sym} — Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

## 4. SHAP Feature Importance

In [ ]:
import pickle
from pathlib import Path

if dfs:
    sym = list(dfs.keys())[0]
    model_path = Path('../models/manual') / f'{sym}_lgbm.pkl'
    if model_path.exists():
        with open(model_path, 'rb') as f:
            lgbm_model = pickle.load(f)
        
        df = dfs[sym]
        feature_cols = [c for c in df.columns if c != 'label']
        X = df[feature_cols].values.astype('float32')
        
        top_feats = lgbm_model.top_features(X, n=20)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        names = list(reversed(list(top_feats.keys())))
        values = list(reversed(list(top_feats.values())))
        ax.barh(names, values, color='steelblue')
        ax.set_xlabel('Mean |SHAP value|')
        ax.set_title(f'{sym} — Top 20 Features by SHAP Importance')
        plt.tight_layout()
        plt.show()
    else:
        print('No trained model found. Run: python main.py --mode train --trainer manual')

## 5. Model Comparison

In [ ]:
from src.evaluation.evaluator import compare_all

comparison_df = compare_all(include_mlflow=False)
if not comparison_df.empty:
    display(comparison_df.style.highlight_max(subset=['Accuracy','Macro F1','Weighted F1'], color='lightgreen'))
else:
    print('No models trained yet.')

## 6. Chronos-2 Zero-Shot Forecast Demo

In [ ]:
# Demo: Chronos-2 forecasting on RELIANCE.NS
try:
    import pandas as pd
    from pathlib import Path
    
    raw_path = Path('../data/raw/RELIANCE.NS.csv')
    if raw_path.exists():
        raw = pd.read_csv(raw_path, index_col='date', parse_dates=True)
        close = raw['close'].dropna().values
        
        from src.models.chronos_model import ChronosModel
        chronos = ChronosModel('RELIANCE.NS')
        proba = chronos.forecast_next(close, context_length=64, n_samples=20)
        
        SIGNAL_MAP = {4: 'Strong Buy', 3: 'Buy', 2: 'Hold', 1: 'Sell', 0: 'Strong Sell'}
        signal = int(np.argmax(proba))
        print(f'Chronos-2 RELIANCE.NS forecast:')
        for i, (name, p) in enumerate(zip(['Strong Sell','Sell','Hold','Buy','Strong Buy'], proba)):
            marker = ' ◄' if i == signal else ''
            print(f'  {name:<12}: {p:.1%}{marker}')
    else:
        print('Raw data not found. Run: python main.py --mode ingest')
except Exception as e:
    print(f'Chronos-2 not available: {e}')
    print('Install: pip install chronos-forecasting==2.2.2')

## 7. Live Sentiment Check

In [ ]:
from src.slm.sentiment import get_live_sentiment

for sym in SYMBOLS[:2]:  # Check first 2 to avoid too many API calls
    sent = get_live_sentiment(sym)
    label = 'positive' if sent['sentiment_score'] > 0.1 else 'negative' if sent['sentiment_score'] < -0.1 else 'neutral'
    print(f'{sym}: score={sent["sentiment_score"]:+.3f}  confidence={sent["sentiment_confidence"]:.2f}  '
          f'news={int(sent["news_count_7d"])}  label={label}')